# Step 2a Global analysis of 670nm excitation TA of WL-PSI of SCy6803

### Package imports

In [ ]:
from glotaran.io import load_scheme
from pyglotaran_extras.compat import convert


def _case_study_convert(native_result, scheme):
    """Project native v0.8 results for legacy plotting while retaining native results."""
    import numpy as np
    import xarray as xr

    compat_result = convert(native_result)
    for dataset_label, dataset in compat_result.data.items():
        if "irf_center" in dataset.coords:
            irf_center = dataset.coords["irf_center"]
            if irf_center.ndim and np.allclose(irf_center, irf_center.values.flat[0]):
                dataset = dataset.drop_vars("irf_center").assign_coords(
                    irf_center=float(irf_center.values.flat[0])
                )
                compat_result.data[dataset_label] = dataset
        optimization_result = native_result.optimization_results[dataset_label]
        global_dimension = optimization_result.meta.global_dimension
        model_dimension = optimization_result.meta.model_dimension
        input_data = optimization_result.input_data
        residual = optimization_result.residuals
        if isinstance(input_data, xr.Dataset):
            input_data = input_data["data"]
        if isinstance(residual, xr.Dataset):
            residual = residual["residual"]
        fitted_data = input_data - residual
        if {"time", "spectral"}.issubset(fitted_data.dims):
            fitted_data = fitted_data.transpose("time", "spectral")
        dataset["fitted_data"] = fitted_data
        data_model = next(
            experiment.datasets[dataset_label]
            for experiment in scheme.experiments.values()
            if dataset_label in experiment.datasets
        )
        weight = xr.ones_like(residual)
        for weight_item in data_model.weights:
            selected = xr.ones_like(residual, dtype=bool)
            if weight_item.global_interval is not None:
                lower, upper = weight_item.global_interval
                selected = selected & (
                    (residual.coords[global_dimension] >= lower)
                    & (residual.coords[global_dimension] <= upper)
                )
            if weight_item.model_interval is not None:
                lower, upper = weight_item.model_interval
                selected = selected & (
                    (residual.coords[model_dimension] >= lower)
                    & (residual.coords[model_dimension] <= upper)
                )
            weight = weight * xr.where(selected, float(weight_item.value), 1.0)
        dataset["weight"] = weight
        dataset["weighted_residual"] = residual * weight
        dataset["clp"] = optimization_result.fit_decomposition.clp.rename(
            amplitude_label="clp_label"
        )
        dataset["matrix"] = optimization_result.fit_decomposition.matrix.rename(
            amplitude_label="clp_label"
        )
        kinetic_elements = [
            element
            for element in optimization_result.elements.values()
            if "compartment" in element.coords
        ]
        if kinetic_elements:
            species_concentration = xr.concat(
                [
                    element["concentrations"].rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            species_concentration = species_concentration.isel(
                species=~species_concentration.get_index("species").duplicated()
            )
            concentration_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_concentration.dims
            ]
            concentration_order.extend(
                dimension
                for dimension in species_concentration.dims
                if dimension not in concentration_order
            )
            dataset["species_concentration"] = species_concentration.transpose(
                *concentration_order
            )
            species_associated_spectra = xr.concat(
                [element["amplitudes"].rename(compartment="species") for element in kinetic_elements],
                dim="species",
            )
            species_associated_spectra = species_associated_spectra.isel(
                species=~species_associated_spectra.get_index("species").duplicated()
            )
            spectra_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_associated_spectra.dims
            ]
            spectra_order.extend(
                dimension
                for dimension in species_associated_spectra.dims
                if dimension not in spectra_order
            )
            dataset["species_associated_spectra"] = species_associated_spectra.transpose(
                *spectra_order
            )
            initial_concentration = xr.concat(
                [
                    element["initial_concentrations"]
                    .isel(activation=0, drop=True)
                    .rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            dataset["initial_concentration"] = initial_concentration.isel(
                species=~initial_concentration.get_index("species").duplicated()
            )
        spectral_elements = [
            element
            for element in optimization_result.elements.values()
            if "shape" in element.coords
        ]
        if spectral_elements:
            species_spectra = xr.concat(
                [
                    element["concentrations"].squeeze(drop=True).rename(shape="species")
                    for element in spectral_elements
                ],
                dim="species",
            )
            spectral_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_spectra.dims
            ]
            spectral_order.extend(
                dimension for dimension in species_spectra.dims if dimension not in spectral_order
            )
            dataset["species_spectra"] = species_spectra.transpose(*spectral_order)
        dataset.attrs["dataset_scale"] = optimization_result.meta.scale
    return compat_result


def _case_study_matrix_markdown(scheme, element_label, compartments=None):
    """Render a symbolic v0.8 kinetic rate map for legacy notebook display cells."""
    import pandas as pd

    element = scheme.library[element_label]
    compartments = list(compartments or element.compartments)
    table = [["" for _ in compartments] for _ in compartments]
    for (to_compartment, from_compartment), rate in element.rates.items():
        if to_compartment in compartments and from_compartment in compartments:
            table[compartments.index(to_compartment)][compartments.index(from_compartment)] = str(rate)
    return pd.DataFrame(table, index=compartments, columns=compartments).to_markdown()


from glotaran.io import load_parameters, save_result
import matplotlib.pyplot as plt
from pyglotaran_extras import plot_overview, plot_data_overview
from pyglotaran_extras import plot_doas
from pyglotaran_extras import plot_fitted_traces, select_plot_wavelengths
from pyglotaran_extras.inspect import show_a_matrixes
from pyglotaran_extras.plotting.style import PlotStyle
from pyglotaran_extras.plotting.style import ColorCode
from cycler import cycler

### Data inspection

In [ ]:
DATA_PATH_670_1 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_reva.ascii'
DATA_PATH_670_2 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revb.ascii'
DATA_PATH_670_3 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revc.ascii'
DATA_PATH_670_4 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revd.ascii'

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH_670_1, nr_of_data_svd_vectors=5, linlog=False, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True, title='670 nm excitation Time Range 1')

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH_670_2, nr_of_data_svd_vectors=5, linlog=False, linthresh=10, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True, title='670 nm excitation Time Range 2')

## Global Analysis

### Model specification

This time the model and parameters are defined in each step, as the model and/or parameters may be tweaked in each step.

### Define the analysis scheme and optimize

A scheme is a collection of a model, parameters and data, along with options for optimization.

In [ ]:
global_scheme = load_scheme('models/global_step1_model_PSI_TA_SCy6803WL670_without_DOAS_v08.yml')
global_scheme_parameters = load_parameters('models/global_step1_parameters_PSI_TA_SCy6803WL670_without_DOAS.csv')
global_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2}
global_scheme_dry_run = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme load=PASS dry_run=PASS')

In [ ]:
global_result_670_native = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme real_fit=PASS')
global_result_670 = _case_study_convert(global_result_670_native, global_scheme)

<sub>For reference, after 14 iterations the final cost is 3.3293e+02</sub>

## Residual analysis of the 670 nm excitation TR1 data

In [ ]:
from custom_plotting import plot_residual_and_svd
(fig, axes) = plot_residual_and_svd([global_result_670.data['670TR1']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

## Plot result for interpretation


In [ ]:
from custom_plotting import plot_concentration_and_spectra
myFRLcolors = ['tab:grey', 'tab:orange', ColorCode.cyan, ColorCode.green, 'm', 'y', 'k', 'r', 'b', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors)
(fig, axes) = plot_concentration_and_spectra([global_result_670.data['670TR1'], global_result_670.data['670TR2']], cycler=custom_cycler, das_cycler=custom_cycler)

In [ ]:
global_scheme = load_scheme('models/global_step2_model_PSI_TA_SCy6803WL670_with_DOAS_v08.yml')
global_scheme_parameters = load_parameters('models/global_step2_parameters_PSI_TA_SCy6803WL670_with_DOAS.csv')
global_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2}
global_scheme_dry_run = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme load=PASS dry_run=PASS')

In [ ]:
global_result670_native = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme real_fit=PASS')
global_result670 = _case_study_convert(global_result670_native, global_scheme)

## Residual analysis of the 670 nm excitation TR1 data

In [ ]:
from custom_plotting import plot_residual_and_svd
(fig, axes) = plot_residual_and_svd([global_result670.data['670TR1']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

## Plot result for interpretation


In [ ]:
myFRLcolors = ['tab:grey', 'tab:orange', ColorCode.cyan, ColorCode.green, 'm', 'y', 'k', 'r', 'b', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors)
(fig, axes) = plot_concentration_and_spectra([global_result670.data['670TR1'], global_result670.data['670TR2']], cycler=custom_cycler, das_cycler=custom_cycler)

To save the results of the optimization we can use the `save_result` command.

Because it saves *everything* it consumes about 20MB of disk space per save.

In [ ]:
save_result(result=global_result670_native, result_path='results/global670/result.yaml', allow_overwrite=True)

### Results and parameters

In [ ]:
global_result670

In [ ]:
global_result670.optimized_parameters

## what to do if the estimated rate constants are not in decreasing order?
then one should sort the estimated rate constants in decreasing order, and repeat the fit with those new starting values

### Amplitude matrices

In [ ]:
show_a_matrixes(global_result670)

## Result plots

<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>

## Fit quality

In [ ]:
global_result_TA = (global_result670.data['670TR1'], global_result670.data['670TR2'])
wavelengths = select_plot_wavelengths(global_result_TA, equidistant_wavelengths=True)
fig_traces = plot_fitted_traces(global_result_TA, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).

## Overview 670 exc

In [ ]:
fig_670TR1 = plot_overview(global_result670.data['670TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']), use_svd_number=True, svd_cycler=PlotStyle().cycler)

In [ ]:
fig_670TR2 = plot_overview(global_result670.data['670TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']), use_svd_number=True, svd_cycler=PlotStyle().cycler)

## Coherent Artifact


In [ ]:
from pyglotaran_extras import plot_coherent_artifact
(fig, axes) = plot_coherent_artifact(global_result670.data['670TR1'], time_range=(-0.3, 0.3), figsize=(10, 4))
axes[0].set_xlabel('Time (ps)')
axes[1].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('')
axes[0].annotate('A', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
fig.tight_layout()

## Overview of the estimated DOAS and phases of 670 nm excitation data

In [ ]:
(fig, axes) = plot_doas(global_result670.data['670TR1'], damped_oscillation=['osc1'], time_range=(-0.3, 0.3), spectral=700, figsize=(15, 4), normalize=False)
axes[0].set_xlabel('Time (ps)')
axes[0].axhline(0, color='k', linewidth=1)
axes[1].set_xlabel('Wavelength (nm)')
axes[2].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('')
axes[1].set_title('DOAS')
axes[0].annotate('C', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)
axes[1].annotate('D', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)
axes[2].annotate('E', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)

# Step 2b Global analysis of 700nm excitation TA of WL-PSI of SCy6803

### Data inspection

In [ ]:
DATA_PATH_700_3 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revc.ascii'
DATA_PATH_700_4 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revd.ascii'

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH_700_3, nr_of_data_svd_vectors=5, linlog=False, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True, title='700 nm excitation Time Range 1')

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH_700_4, nr_of_data_svd_vectors=5, linlog=False, linthresh=10, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True, title='700 nm excitation Time Range 2')

## Global Analysis

### Model specification

The model and parameters are defined in each step, as the model and/or parameters may be tweaked in each step.

### Create scheme and optimize it

In [ ]:
global_scheme = load_scheme('models/global_step3_model_PSI_TA_SCy6803WL700_v08.yml')
global_scheme_parameters = load_parameters('models/global_step3_parameters_PSI_TA_SCy6803WL700.csv')
global_scheme_datasets = {'700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
global_scheme_dry_run = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme load=PASS dry_run=PASS')

In [ ]:
global_result700_native = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=15, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme real_fit=PASS')
global_result700 = _case_study_convert(global_result700_native, global_scheme)

To save the results of the optimization we can use the `save_result` command.

Because it saves *everything* it consumes about 50MB of disk space per save.

In [ ]:
save_result(result=global_result700_native, result_path='results/global700/result.yaml', allow_overwrite=True)

### Results and parameters

In [ ]:
global_result700

In [ ]:
global_result700.optimized_parameters

## what to do if the estimated rate constants are not in decreasing order?
then one should sort the estimated rate constants in decreasing order, and repeat the fit with those new starting values

### Amplitude matrices

In [ ]:
show_a_matrixes(global_result700)

## Result plots

<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>

## Fit quality

In [ ]:
global_result_TA_700 = (global_result700.data['700TR1'], global_result700.data['700TR2'])
wavelengths = select_plot_wavelengths(global_result_TA_700, equidistant_wavelengths=True)
fig_traces_700 = plot_fitted_traces(global_result_TA_700, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).

## Overview 700 exc

In [ ]:
custom_cycler = cycler(color=['k', 'r', 'g', 'tab:purple'])
fig_700TR1 = plot_overview(global_result700.data['700TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=custom_cycler, use_svd_number=True, svd_cycler=PlotStyle().cycler)

In [ ]:
fig_700TR = plot_overview(global_result700.data['700TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=10, cycler=custom_cycler, use_svd_number=True, svd_cycler=PlotStyle().cycler)

## Coherent Artifact


In [ ]:
from pyglotaran_extras import plot_coherent_artifact
(fig, axes) = plot_coherent_artifact(global_result700.data['700TR1'], time_range=(-0.3, 0.3), figsize=(10, 4))
axes[0].set_xlabel('Time (ps)')
axes[1].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('')
axes[0].annotate('A', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
fig.tight_layout()

## Plot result for interpretation


In [ ]:
custom_cycler = cycler(color=['k', 'r', 'g', 'tab:purple'])
(fig, axes) = plot_concentration_and_spectra([global_result700.data['700TR1'], global_result700.data['700TR2']], cycler=custom_cycler, das_cycler=custom_cycler)

In [ ]:
from custom_plotting import plot_final_and_diff_EADS
(fig_EADSdiff, _) = plot_final_and_diff_EADS(global_result670.data['670TR1'], global_result700.data['700TR1'], scale=0.85)

## Residual analysis of the 700 nm excitation TR1 data

In [ ]:
(fig, axes) = plot_residual_and_svd([global_result700.data['700TR1']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)


# Step 2c Simultaneous global analysis of TA of WL-PSI of SCy6803

### Create scheme and optimize it

In [ ]:
global_scheme = load_scheme('models/global_step4_model_PSI_TA_SCy6803WL670and700_v08.yml')
global_scheme_parameters = load_parameters('models/global_step4_parameters_PSI_TA_SCy6803WL670and700.csv')
global_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
global_scheme_dry_run = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=5, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme load=PASS dry_run=PASS')

In [ ]:
global_result_670_700_native = global_scheme.optimize(parameters=global_scheme_parameters, datasets=global_scheme_datasets, maximum_number_function_evaluations=5, raise_exception=True)
print('MIGRATION_VALIDATION scheme=global_scheme real_fit=PASS')
global_result_670_700 = _case_study_convert(global_result_670_700_native, global_scheme)

To save the results of the optimization we can use the `save_result` command.

Because it saves *everything* it consumes about 40MB of disk space per save.

In [ ]:
save_result(result=global_result_670_700_native, result_path='results/global670and700/result.yaml', allow_overwrite=True)

### Results and parameters

In [ ]:
global_result_670_700

In [ ]:
global_result_670_700.optimized_parameters

### Amplitude matrices

In [ ]:
show_a_matrixes(global_result_670_700)

## Result plots

<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>

## Fit quality

In [ ]:
global_result_TA = (global_result_670_700.data['670TR1'], global_result_670_700.data['670TR2'], global_result_670_700.data['700TR1'], global_result_670_700.data['700TR2'])
wavelengths = select_plot_wavelengths(global_result_TA, equidistant_wavelengths=True)
fig_TA = plot_fitted_traces(global_result_TA, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).
To show to make a manual selection of traces, and 'dress up the plot' see the code below, which reproduces Figure 2 of the iScience paper.

In [ ]:
from pyglotaran_extras.plotting.style import ColorCode as cc
from custom_plotting import plot_fitted_traces_iscience
(fig, ax_) = plot_fitted_traces_iscience(global_result_TA, [685, 700, 720, 760], linlog=True, linthresh=1, axes_shape=(2, 2), figsize=(6, 4), title='', per_axis_legend=True, cycler=cycler(color=[cc.grey, cc.black, cc.grey, cc.black, cc.orange, cc.red, cc.orange, cc.red]))

## Overview 670 exc

In [ ]:
from pyglotaran_extras.plotting.style import ColorCode
fig_670_TR1 = plot_overview(global_result_670_700.data['670TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=['tab:grey', 'tab:orange', ColorCode.cyan, ColorCode.green, 'm', 'y', 'k', 'r', 'b', 'tab:purple']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

## Residual analysis of the 670 nm excitation TR1 data

In [ ]:
(fig, axes) = plot_residual_and_svd([global_result_670_700.data['670TR1']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

In [ ]:
global_result_670_700.data['670TR2']

## Residual analysis of all data

In [ ]:
from custom_plotting import plot_svd_of_residual
(fig, axes) = plot_svd_of_residual([global_result_670_700.data['670TR1'], global_result_670_700.data['670TR2'], global_result_670_700.data['700TR1'], global_result_670_700.data['700TR2']], linlog=True, linthresh=1, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

In [ ]:
global_result_670_700.data['700TR2'].residual.plot(x='time')

In [ ]:
global_result_670_700.data['700TR2'].weighted_residual.plot(x='time')

In [ ]:
global_result_670_700.data['700TR1'].weighted_residual.plot(x='time')

to zoom in on the largest values

In [ ]:
global_result_670_700.data['700TR2'].residual_right_singular_vectors.isel(spectral=slice(56, 72), right_singular_value_index=0).plot()

In [ ]:
global_result_670_700.data['700TR2'].weighted_residual_right_singular_vectors.isel(spectral=slice(56, 72), right_singular_value_index=0).plot()

In [ ]:
(fig, axis) = plt.subplots(1, 1, figsize=(3, 2))
global_result_670_700.data['700TR1'].data.plot(x='time', ax=axis, vmin=-5, vmax=5, cmap='seismic')
axis.set_xlabel('Time (ps)')
axis.set_ylabel('Wavelength (nm)')

In [ ]:
(fig, axes) = plot_svd_of_residual([global_result_670_700.data['670TR1'], global_result_670_700.data['670TR2'], global_result_670_700.data['700TR1'], global_result_670_700.data['700TR2']], linlog=True, linthresh=1, index=1)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[0].set_title('residual 2nd LSV')
axes[1].set_title('residual 2nd RSV')

In [ ]:
_ = plot_overview(global_result_670_700.data['670TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=['tab:grey', 'tab:orange', ColorCode.cyan, ColorCode.green, 'm', 'y', 'k', 'r', 'b', 'tab:purple']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

## Overview 700 exc

In [ ]:
_ = plot_overview(global_result_670_700.data['700TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=['k', 'r', 'g', 'tab:purple']), use_svd_number=True, svd_cycler=PlotStyle().cycler)

In [ ]:
_ = plot_overview(global_result_670_700.data['700TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=['k', 'r', 'g', 'tab:purple']), use_svd_number=True, svd_cycler=PlotStyle().cycler)

## Residual analysis of the 700 nm excitation TR2 data

In [ ]:
(fig, axes) = plot_residual_and_svd([global_result_670_700.data['700TR2']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

Note that the above figure is different from Fig.15 in the paper, where the unweighted residual is depicted in panel A.

## Plot result for interpretation


In [ ]:
myFRLcolors = ['tab:grey', 'tab:orange', ColorCode.cyan, ColorCode.green, 'm', 'y', 'k', 'r', 'b', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors)
(fig, axes) = plot_concentration_and_spectra([global_result_670_700.data['670TR1'], global_result_670_700.data['670TR2']], cycler=custom_cycler)

In [ ]:
myFRLcolors = ['k', 'r', 'g', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors)
(fig, axes) = plot_concentration_and_spectra([global_result_670_700.data['700TR1'], global_result_670_700.data['700TR2']], cycler=custom_cycler, labels=('D', 'E', 'F'))

## Coherent Artifact


In [ ]:
from pyglotaran_extras import plot_coherent_artifact
(fig, axes) = plot_coherent_artifact(global_result_670_700.data['670TR1'], time_range=(-0.3, 0.3), figsize=(10, 4))
axes[0].set_xlabel('Time (ps)')
axes[1].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('')
axes[0].annotate('A', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
fig.tight_layout()

In [ ]:
from pyglotaran_extras import plot_coherent_artifact
(fig, axes) = plot_coherent_artifact(global_result_670_700.data['700TR1'], time_range=(-0.3, 0.3), figsize=(10, 4))
axes[0].set_xlabel('Time (ps)')
axes[1].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('')
axes[0].annotate('A', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(0.02, 0.9), xycoords='axes fraction', fontsize=16)
fig.tight_layout()

## Overview of the estimated DOAS and phases of 670 nm excitation data

In [ ]:
from pyglotaran_extras import plot_doas
from pyglotaran_extras.plotting.style import ColorCode
(fig, axes) = plot_doas(global_result_670_700.data['670TR1'], damped_oscillation=['osc1'], time_range=(-0.3, 0.3), spectral=700, figsize=(15, 4), normalize=False)
axes[0].set_xlabel('Time (ps)')
axes[0].axhline(0, color='k', linewidth=1)
axes[1].set_xlabel('Wavelength (nm)')
axes[2].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('')
axes[1].set_title('DOAS')
axes[0].annotate('C', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)
axes[1].annotate('D', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)
axes[2].annotate('E', xy=(0.01, 0.89), xycoords='axes fraction', fontsize=16)

In [ ]:
from glotaran.io import save_dataset
from glotaran.utils.io import create_clp_guide_dataset
for species in global_result_670_700.data['670TR2'].species:
    clp_guide = create_clp_guide_dataset(global_result_670_700.data['670TR2'], species.item())
    string_in_string = 'guide/global670and700_670TR2_clp_{}.ascii'.format(species.item())
    save_dataset(clp_guide.data, string_in_string, allow_overwrite=True)